In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

print(os.listdir("/content/drive/MyDrive"))

['Bicycle Management', 'Aquarium_Fish', 'Colab Notebooks', 'flower.ipynb', 'Croplens']


In [ ]:
import os

flower_path = "/content/drive/MyDrive/Croplens"

for item in os.listdir(flower_path):
    print(item)

Raw Data set
Dataset


In [ ]:
import os

base_path = "/content/drive/MyDrive/Croplens"

print("Inside Flower:")
print(os.listdir(base_path))

Inside Flower:
['Raw Data set', 'Dataset']


In [ ]:
print("\nCroplens:")
print(os.listdir(os.path.join(base_path, "Dataset")))


Croplens:
['Allium_Cepa', 'Black_cumin', 'Sweet_pea']


In [ ]:
import os

base_path = "/content/drive/MyDrive/Croplens"

image_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            image_files.append(os.path.join(root, file))

print("Total images found:", len(image_files))
print("Sample:", image_files[:5])

Total images found: 1270
Sample: ['/content/drive/MyDrive/Croplens/Raw Data set/testing/allium_cepa/IMG-20260206-WA0082.jpg', '/content/drive/MyDrive/Croplens/Raw Data set/testing/allium_cepa/IMG-20260206-WA0091.jpg', '/content/drive/MyDrive/Croplens/Raw Data set/testing/allium_cepa/IMG-20260206-WA0092.jpg', '/content/drive/MyDrive/Croplens/Raw Data set/testing/allium_cepa/IMG-20260206-WA0101.jpg', '/content/drive/MyDrive/Croplens/Raw Data set/testing/allium_cepa/IMG-20260206-WA0102.jpg']


In [ ]:
dataset_path = "/content/drive/MyDrive/Croplens/Dataset"

In [ ]:
base_path = "/content/drive/MyDrive/Croplens/Dataset"

In [ ]:
!pip install tensorflow

In [ ]:
import os

base_path = "/content/drive/MyDrive/Croplens/Dataset"

class_counts = {}
total = 0

for root, dirs, files in os.walk(base_path):
    # detect image files in each folder
    images = [f for f in files if f.lower().endswith(('.jpg','.jpeg','.png'))]

    if len(images) > 0:
        class_name = os.path.basename(root)
        class_counts[class_name] = class_counts.get(class_name, 0) + len(images)
        total += len(images)

print("Total images found:", total)
print("\nSample class counts:")
for k,v in list(class_counts.items())[:10]:
    print(k, ":", v)

Total images found: 640

Sample class counts:
Allium_Cepa : 171
Black_cumin : 252
Sweet_pea : 217


In [ ]:
import tensorflow as tf

dataset_base_path = "/content/drive/MyDrive/Croplens/Dataset"

# Define a validation split ratio (e.g., 20% for validation)
validation_split_ratio = 0.2
seed_value = 123 # For reproducibility

train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_base_path,
    validation_split=validation_split_ratio,
    subset="training",
    seed=seed_value,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_base_path,
    validation_split=validation_split_ratio,
    subset="validation",
    seed=seed_value,
    image_size=(224,224),
    batch_size=32
)

Found 640 files belonging to 3 classes.
Using 512 files for training.
Found 640 files belonging to 3 classes.
Using 128 files for validation.


**Data Preprocessing**

In [ ]:
!pip install ultralytics
!pip install tensorflow scikit-learn seaborn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.5 MB/s eta 0:00:00


**densenet121**

In [ ]:
# ==========================================
# Google Colab version of densenet121.py
# ==========================================

# Install required library (not preinstalled in Colab)
!pip install python-docx

# Imports
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from docx import Document
from docx.shared import Inches

# ==========================================
# Mount Google Drive
# ==========================================

# Paths (update these according to your dataset in Drive)
input_path = "/content/drive/MyDrive/Croplens/Dataset"   # dataset folder
output_path = "/content/drive/MyDrive/Croplens/Densenet"
os.makedirs(output_path, exist_ok=True)

# Parameters
EPOCHS = 25
BATCH_SIZE = 32
LEARNING_RATE = 0.0001

# Create Document
doc = Document()
doc.add_heading('Classification Report for DenseNet121', 0)

# ==========================================
# DenseNet121 model builder
# ==========================================
def create_densenet_model(input_shape, num_classes):
    base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=input_shape)

    # Fine-tuning last 50 layers
    for layer in base_model.layers[:-50]:
        layer.trainable = False
    for layer in base_model.layers[-50:]:
        layer.trainable = True

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(2048, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

# ==========================================
# Data pipeline
# ==========================================
def process_data(input_path, img_size, batch_size):
    datagen = ImageDataGenerator(
        preprocessing_function=tf.keras.applications.densenet.preprocess_input,
        validation_split=0.2,
        rotation_range=30,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode="nearest",
        brightness_range=[0.8, 1.2],
        channel_shift_range=20.0
          )

    train_gen = datagen.flow_from_directory(
        input_path,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        subset='training',
        shuffle=True
    )

    val_gen = datagen.flow_from_directory(
        input_path,
        target_size=img_size,
        batch_size=batch_size,
        class_mode='categorical',
        subset='validation',
        shuffle=False
    )

    return train_gen, val_gen

# Image size for DenseNet121
img_size = (224, 224) # Changed from 299, 299 to reduce memory usage

# Prepare data
train_gen, val_gen = process_data(input_path, img_size, BATCH_SIZE)

# Create model
model = create_densenet_model((img_size[0], img_size[1], 3), train_gen.num_classes)

# Compile
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
checkpoint = ModelCheckpoint(os.path.join(output_path, 'DenseNet121_best_model.keras'),
                             monitor='val_accuracy',
                             save_best_only=True,
                             mode='max',
                             verbose=1)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=3,
                              verbose=1, min_lr=1e-6)

# ==========================================
# Training
# ==========================================
history = model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=[checkpoint, reduce_lr]
)

# ==========================================
# Fine-Tuning (This part was not executed, but contained a syntax error)
# ==========================================
# base_model.trainable = True # Uncomment and adjust layers for fine-tuning if desired
# for layer in base_model.layers[:-30]:
#     layer.trainable = False

# model.compile(optimizer=Adam(learning_rate=LEARNING_RATE * 10), # Fixed syntax error
#               loss='categorical_crossentropy',
#               metrics=['accuracy'])

# fine_tune_epochs = 10
# model.fit(
#     train_gen,
#     epochs=fine_tune_epochs,
#     validation_data=val_gen,
#     callbacks=[checkpoint, reduce_lr]
# )

# ==========================================
# Accuracy & Loss Plots
# ==========================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title("DenseNet121 Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
plt.title("DenseNet121 Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

acc_loss_path = os.path.join(output_path, 'DenseNet121_accuracy_loss_curve.png')
plt.savefig(acc_loss_path)
plt.close()

# ==========================================
# Evaluation & Predictions
# ==========================================
model.load_weights(os.path.join(output_path, 'DenseNet121_best_model.keras'))
eval_result = model.evaluate(val_gen)

Y_pred = model.predict(val_gen)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_gen.classes

report = classification_report(y_true, y_pred, target_names=list(train_gen.class_indices.keys()))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_path = os.path.join(output_path, 'DenseNet121_confusion_matrix.png')

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=train_gen.class_indices.keys(),
            yticklabels=train_gen.class_indices.keys(),
            annot_kws={"size": 14})
plt.xticks(rotation=45, fontsize=12)
plt.yticks(rotation=0, fontsize=12)
plt.xlabel('Predicted', fontsize=14)
plt.ylabel('Actual', fontsize=14)
plt.title('DenseNet121 Confusion Matrix', fontsize=16)
plt.tight_layout()
plt.savefig(cm_path)
plt.close()

# ==========================================
# Word Report
# ==========================================
doc.add_heading('DenseNet121 Performance', level=1)
doc.add_paragraph(f"Final Validation Loss: {eval_result[0]:.4f}")
doc.add_paragraph(f"Final Validation Accuracy: {eval_result[1]:.4f}")

doc.add_heading('Classification Report', level=2)
doc.add_paragraph(report)

doc.add_heading('Training Accuracy and Loss', level=2)
doc.add_picture(acc_loss_path, width=Inches(6))

doc.add_heading('Confusion Matrix', level=2)
doc.add_picture(cm_path, width=Inches(6))

doc.save(os.path.join(output_path, 'DenseNet121_classification_report.docx'))

print("✅ DenseNet121 model training and report generation completed! Results saved in Google Drive.")

Found 513 images belonging to 3 classes.
Found 127 images belonging to 3 classes.
Epoch 1/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.6764 - loss: 0.8120 
Epoch 1: val_accuracy improved from None to 0.66142, saving model to /content/drive/MyDrive/Croplens/Densenet/DenseNet121_best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Croplens/Densenet/DenseNet121_best_model.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 304s 17s/step - accuracy: 0.8402 - loss: 0.3987 - val_accuracy: 0.6614 - val_loss: 0.5748 - learning_rate: 1.0000e-04
Epoch 2/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11s/step - accuracy: 0.9935 - loss: 0.0238 
Epoch 2: val_accuracy improved from 0.66142 to 0.81890, saving model to /content/drive/MyDrive/Croplens/Densenet/DenseNet121_best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Croplens/Densenet/DenseNet121_best_model.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 256s 15s/step - accuracy: 0.9942 - loss: 0.0216 - val_accuracy: 0.8189 - val_loss: 0.

In [ ]:
from tensorflow.keras.applications import DenseNet121

# Load pretrained DenseNet121 (ImageNet weights)
model = DenseNet121(
    weights='imagenet',
    include_top=True
)

# Save as .keras
model.save("DenseNet121.keras")

33188688/33188688 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
from google.colab import files

files.download("DenseNet121.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**InceptionV3**

In [ ]:
 # ==========================================
# Google Colab version of InceptionV3 code
# ==========================================

# Install required libraries
!pip install python-docx seaborn

# Imports
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from docx import Document
from docx.shared import Inches

# ==========================================
# Mount Google Drive
# ==========================================


# Paths (Update these to your dataset location in Drive)
input_path = "/content/drive/MyDrive/Croplens/Dataset"   # dataset folder
output_path = "/content/drive/MyDrive/Croplens/InceptionV3_result"
os.makedirs(output_path, exist_ok=True)

# Parameters
IMG_SIZE = (299, 299)
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 0.0001

# ==========================================
# Data Augmentation and Preprocessing
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest"
)

train_generator = train_datagen.flow_from_directory(
    input_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    input_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ==========================================
# Load Pretrained InceptionV3
# ==========================================
base_model = InceptionV3(weights='imagenet', include_top=False, input_shape=(299, 299, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

# Compile
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Callbacks
checkpoint = ModelCheckpoint(os.path.join(output_path, 'best_model.keras'),
                             monitor='val_accuracy',
                             save_best_only=True,
                             mode='max',
                             verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# ==========================================
# Train the Model
# ==========================================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=[checkpoint, early_stop]
)

# ==========================================
# Plot Accuracy & Loss
# ==========================================
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

accuracy_loss_curve_path = os.path.join(output_path,'accuracy_loss_curve.png')
plt.savefig(accuracy_loss_curve_path)
plt.close()

# ==========================================
# Evaluation & Predictions
# ==========================================
model.load_weights(os.path.join(output_path, 'best_model.keras'))
eval_result = model.evaluate(validation_generator)

Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = validation_generator.classes

report = classification_report(y_true, y_pred, target_names=list(train_generator.class_indices.keys()))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
confusion_matrix_path = os.path.join(output_path, 'confusion_matrix.png')

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=train_generator.class_indices.keys(),
            yticklabels=train_generator.class_indices.keys())
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig(confusion_matrix_path)
plt.close()

# ==========================================
# Word Report
# ==========================================
doc = Document()
doc.add_heading('Classification Report - InceptionV3', 0)

doc.add_heading('Model Performance', level=1)
doc.add_paragraph(f"Final Validation Loss: {eval_result[0]:.4f}")
doc.add_paragraph(f"Final Validation Accuracy: {eval_result[1]:.4f}")

doc.add_heading('Classification Metrics', level=1)
doc.add_paragraph(report)

doc.add_heading('Training History', level=1)
doc.add_picture(accuracy_loss_curve_path, width=Inches(6))

doc.add_heading('Confusion Matrix', level=1)
doc.add_picture(confusion_matrix_path, width=Inches(6))

doc.save(os.path.join(output_path, 'classification_report.docx'))

print("✅ Training completed and results saved in Google Drive!")

Found 513 images belonging to 3 classes.
Found 127 images belonging to 3 classes.
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14s/step - accuracy: 0.5805 - loss: 0.9758 
Epoch 1: val_accuracy improved from None to 0.99213, saving model to /content/drive/MyDrive/Croplens/InceptionV3_result/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Croplens/InceptionV3_result/best_model.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 323s 20s/step - accuracy: 0.7115 - loss: 0.6951 - val_accuracy: 0.9921 - val_loss: 0.1968
Epoch 2/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13s/step - accuracy: 0.9762 - loss: 0.1817 
Epoch 2: val_accuracy improved from 0.99213 to 1.00000, saving model to /content/drive/MyDrive/Croplens/InceptionV3_result/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Croplens/InceptionV3_result/best_model.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 281s 16s/step - accuracy: 0.9844 - loss: 0.1418 - val_accuracy: 1.0000

In [1]:

from tensorflow.keras.applications import InceptionV3

# Load pretrained InceptionV3 (ImageNet weights)
model = InceptionV3(
    weights='imagenet',
    include_top=True
)

# Save as .keras
model.save("InceptionV3.keras")

96112376/96112376 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [2]:
from google.colab import files

files.download("InceptionV3.keras")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**ResNet50**

In [4]:
# ==========================================
# ResNet50 Image Classification (Google Colab version)
# ==========================================

# Install dependencies
!pip install python-docx seaborn

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns
from docx import Document
from docx.shared import Inches

# ==========================================
# Mount Google Drive
# ==========================================


# Paths (Update to your dataset location in Drive)
input_path = "/content/drive/MyDrive/Croplens/Dataset"
output_path = "/content/drive/MyDrive/Croplens/ResNet50_output"
os.makedirs(output_path, exist_ok=True)

# Parameters
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 0.0001

# ==========================================
# Data Augmentation
# ==========================================
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    fill_mode="nearest"
)

train_generator = train_datagen.flow_from_directory(
    input_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    input_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

# ==========================================
# Class Weights
# ==========================================
classes = list(train_generator.class_indices.keys())
labels = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)
class_weights = dict(enumerate(class_weights))

# ==========================================
# Load ResNet50 Base
# ==========================================
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

# Top Layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)

# Compile
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# ==========================================
# Callbacks
# ==========================================
checkpoint = ModelCheckpoint(os.path.join(output_path, 'best_model.keras'),
                             monitor='val_loss', save_best_only=True, mode='min', verbose=1)

early_stop = EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

# ==========================================
# Training (Frozen Base)
# ==========================================
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    class_weight=class_weights,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

# ==========================================
# Fine-Tuning
# ==========================================
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=Adam(learning_rate=LEARNING_RATE * 10),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

fine_tune_epochs = 10
model.fit(
    train_generator,
    epochs=fine_tune_epochs,
    validation_data=validation_generator,
    class_weight=class_weights,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

# ==========================================
# Plot Accuracy & Loss
# ==========================================
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', marker='o')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', marker='o')
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', marker='o')
plt.plot(history.history['val_loss'], label='Validation Loss', marker='o')
plt.title("Model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

accuracy_loss_curve_path = os.path.join(output_path, 'accuracy_loss_curve.png')
plt.savefig(accuracy_loss_curve_path)
plt.close()

# ==========================================
# Load Best Model
# ==========================================
model.load_weights(os.path.join(output_path, 'best_model.keras'))

# Evaluation
eval_result = model.evaluate(validation_generator)
Y_pred = model.predict(validation_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = validation_generator.classes
report = classification_report(y_true, y_pred, target_names=classes)

# ==========================================
# Confusion Matrix
# ==========================================
cm = confusion_matrix(y_true, y_pred)
confusion_matrix_path = os.path.join(output_path, 'confusion_matrix.png')
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes,
            yticklabels=classes)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.savefig(confusion_matrix_path)
plt.close()

# ==========================================
# Word Report
# ==========================================
doc = Document()
doc.add_heading('Classification Report - ResNet50', 0)

doc.add_heading('Model Performance', level=1)
doc.add_paragraph(f"Final Validation Loss: {eval_result[0]:.4f}")
doc.add_paragraph(f"Final Validation Accuracy: {eval_result[1]:.4f}")

doc.add_heading('Classification Metrics', level=1)
doc.add_paragraph(report)

doc.add_heading('Training History', level=1)
doc.add_picture(accuracy_loss_curve_path, width=Inches(6))

doc.add_heading('Confusion Matrix', level=1)
doc.add_picture(confusion_matrix_path, width=Inches(6))

doc.save(os.path.join(output_path, 'classification_report_resnet.docx'))
print("✅ ResNet50 training complete and report saved in Google Drive.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.9 MB/s eta 0:00:00


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Croplens/Dataset'

In [ ]:
from tensorflow.keras.applications import ResNet50

# Load pretrained ResNet50 (ImageNet weights)
model = ResNet50(
    weights='imagenet',
    include_top=True
)

# Save as .keras
model.save("ResNet50.keras")

In [ ]:
from google.colab import files

files.download("ResNet50.keras")

**YOLOv8 Classification**

Train

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")

results = model.train(
    data="/content/dataset",
    epochs=10,
    imgsz=224
)

Validate:

In [ ]:
metrics = model.val()
print(metrics.top1)
print(metrics.top5)

**Confusion Matrix Function**

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(model, test_ds, class_names):

    y_true = np.concatenate(
        [y.numpy() for x,y in test_ds]
    )

    y_pred = np.argmax(
        model.predict(test_ds),
        axis=1
    )

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10,8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
plot_confusion_matrix(
    efficientnet,
    test_ds,
    class_names
)

**Accuracy Graph**

In [ ]:
import matplotlib.pyplot as plt

def plot_accuracy(history, title):

    plt.figure(figsize=(8,5))

    plt.plot(
        history.history['accuracy'],
        label='Train'
    )

    plt.plot(
        history.history['val_accuracy'],
        label='Validation'
    )

    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

In [ ]:
plot_accuracy(hist_eff, "EfficientNet")
plot_accuracy(hist_res, "ResNet50")
plot_accuracy(hist_mobile, "MobileNetV2")